# G2 Network Simulator — Performance Analysis

Analyze **why** G2 is slow by examining internal RunTracker statistics:
- Per-function **average time per call** and how it scales with NPU count
- Whether slowness comes from **more calls** or **each call getting slower**
- `flows_added`, `total_hops` (and later `total_data_sent`) vs workload parameters
- Cache efficiency and structural complexity metrics

> **Note:** Untracked time (wall − tracked) is AstraSim overhead, not G2. All analysis here uses only G2's tracked execution time.

In [ ]:
import os, re, json, warnings, time
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression
import plotly.io as pio
from typing import Dict, List

warnings.filterwarnings('ignore')
pio.renderers.default = 'plotly_mimetype'

BASE_OUTPUT_DIR = '/app/astra-sim/upc/output/comparison_run/'
EXPERIMENT = 'experiment8___'
TOPOLOGY = 'FoldedClosECMP1024'

PARAM_COLS = ['npu_count', 'd_model', 'num_stacks', 'seq_len', 'batch', 'micro_batch', 'dp', 'tp', 'pp', 'weight_sharded']

print('Libraries loaded.')

Libraries loaded.


## 1. Data Collection

Walk all G2 runs under `experiment8/FoldedClosECMP1024`, parse:
- Workload parameters from directory names
- Wall-clock time from `run_summary.txt`
- RunTracker JSON: `execution_times`, `network_stats`, `cache_stats`

In [161]:
# --- Helper Functions (same as sim_time_scaling notebook) ---

def parse_config(file_path: str) -> Dict[str, str]:
    """Parses a 'key: value' or 'key = value' configuration file."""
    params = {}
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                parts = re.split(r'[:=]', line, 1)
                if len(parts) == 2:
                    key, value = parts
                    params[key.strip().lower()] = value.strip()
    except Exception:
        pass
    return params


def parse_runtime(time_str: str) -> float:
    """Parses 'H:MM:SS.ffffff' into total seconds."""
    if not time_str:
        return 0.0
    try:
        parts = time_str.split(':')
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    except (ValueError, IndexError):
        return 0.0


def parse_workload_params(workload_name: str) -> dict:
    """Parse model/parallelism params from workload directory name.
    Format: d{dmodel}_L{layers}_seq{seq}_b{batch}_mb{mb}_{dp}_{tp}_{sp}_{pp}_{ws}
    """
    m = re.match(
        r'd(\d+)_L(\d+)_seq(\d+)_b(\d+)_mb(\d+)_(\d+)_(\d+)_(\d+)_(\d+)_(\d+)',
        workload_name
    )
    if not m:
        return {}
    return {
        'd_model': int(m.group(1)),
        'num_stacks': int(m.group(2)),
        'seq_len': int(m.group(3)),
        'batch': int(m.group(4)),
        'micro_batch': int(m.group(5)),
        'dp': int(m.group(6)),
        'tp': int(m.group(7)),
        'sp': int(m.group(8)),
        'pp': int(m.group(9)),
        'weight_sharded': int(m.group(10)),
    }

print('Helpers defined.')

Helpers defined.


In [162]:
# --- Collect G2 RunTracker data ---

topo_base = os.path.join(BASE_OUTPUT_DIR, EXPERIMENT, TOPOLOGY)
rows = []           # one row per (npu_count, workload, run) with summary stats
exec_time_rows = [] # one row per (npu_count, workload, run, section)

for npu_dir in sorted(os.listdir(topo_base)):
    npu_path = os.path.join(topo_base, npu_dir)
    if not os.path.isdir(npu_path) or not npu_dir.startswith('npu_'):
        continue
    npu_count = int(npu_dir.split('_')[1])

    for workload_name in os.listdir(npu_path):
        workload_path = os.path.join(npu_path, workload_name)
        if not os.path.isdir(workload_path):
            continue
        wparams = parse_workload_params(workload_name)
        if not wparams:
            continue

        for run_dir_name in os.listdir(workload_path):
            if not run_dir_name.startswith('run_g2'):
                continue
            run_path = os.path.join(workload_path, run_dir_name)
            g2_path = os.path.join(run_path, 'g2')
            if not os.path.isdir(g2_path):
                continue

            # Find runtracker JSON
            rt_files = [f for f in os.listdir(g2_path) if f.endswith('_runtracker.json')]
            if not rt_files:
                continue

            rt_path = os.path.join(g2_path, rt_files[0])
            try:
                with open(rt_path) as f:
                    rt = json.load(f)
            except Exception:
                continue

            # Wall-clock from run_summary.txt
            summary_params = parse_config(os.path.join(run_path, 'run_summary.txt'))
            wall_time_sec = parse_runtime(summary_params.get('total runtime', '0'))
            if wall_time_sec <= 0:
                wall_time_sec = rt.get('wall_total_s', 0)

            net_stats = rt.get('network_stats', {})
            cache_stats = rt.get('cache_stats', {})

            row_base = {
                'workload': workload_name,
                'npu_count': npu_count,
                'run': run_dir_name,
                'wall_time_sec': wall_time_sec,
                'g2_wall_s': rt.get('wall_total_s', 0),
                'tracked_s': rt.get('tracked_execution_time_s', 0),
                'untracked_s': rt.get('untracked_time_s', 0),
                # Network stats
                'flows_added': net_stats.get('flows_added', 0),
                'flows_removed': net_stats.get('flows_removed', 0),
                'total_hops': net_stats.get('total_hops', 0),
                'get_next_messages_calls': net_stats.get('get_next_messages_calls', 0),
                'remove_messages_calls': net_stats.get('remove_messages_calls', 0),
                'peak_transmitting_flows': net_stats.get('peak_transmitting_flows', 0),
                'peak_propagating_flows': net_stats.get('peak_propagating_flows', 0),
                'peak_active_flow_routes': net_stats.get('peak_active_flow_routes', 0),
                'json_recreations': net_stats.get('Json recreation', 0),
                'cache_hits': net_stats.get('Cache Hits', 0),
                'threshold_recreates': net_stats.get('threshold_recreate_count',
                                        net_stats.get('Threshold recreate count', 0)),
                # Cache stats
                'superset_hits': cache_stats.get('superset_hits', 0),
                'superset_misses': cache_stats.get('superset_misses', 0),
                **wparams,
            }
            rows.append(row_base)

            # Per-section execution times
            for et in rt.get('execution_times', []):
                exec_time_rows.append({
                    'workload': workload_name,
                    'npu_count': npu_count,
                    'run': run_dir_name,
                    'section': et['section'],
                    'total_s': et['total_s'],
                    'calls': et['calls'],
                    'avg_ms': et['avg_ms'],
                    'pct': et['pct'],
                    **wparams,
                })

print(f'Collected {len(rows)} G2 runs, {len(exec_time_rows)} section-level records')

Collected 9 G2 runs, 327 section-level records


In [163]:
# --- Build DataFrames ---

if not rows:
    raise SystemExit('No G2 runtracker data found. Run simulations first.')

df = pd.DataFrame(rows)
et_df = pd.DataFrame(exec_time_rows)

# Log-transform key columns
eps = 1e-9
for col in ['g2_wall_s', 'flows_added', 'total_hops', 'get_next_messages_calls',
            'npu_count', 'd_model', 'num_stacks', 'seq_len', 'batch', 'micro_batch']:
    if col in df.columns:
        df[f'log_{col}'] = np.log10(df[col].clip(lower=eps))

print(f'Runs: {len(df)}  |  NPU counts: {sorted(df.npu_count.unique())}  |  Workloads: {df.workload.nunique()}')
print(f'\nG2 wall time range (s):')
print(df.groupby('npu_count')['g2_wall_s'].describe().round(2))

Runs: 9  |  NPU counts: [2, 4, 8, 16, 32, 64, 128, 256, 512]  |  Workloads: 9

G2 wall time range (s):
           count    mean  std     min     25%     50%     75%     max
npu_count                                                            
2            1.0    0.02  NaN    0.02    0.02    0.02    0.02    0.02
4            1.0    8.23  NaN    8.23    8.23    8.23    8.23    8.23
8            1.0    0.09  NaN    0.09    0.09    0.09    0.09    0.09
16           1.0    0.32  NaN    0.32    0.32    0.32    0.32    0.32
32           1.0   17.91  NaN   17.91   17.91   17.91   17.91   17.91
64           1.0   17.60  NaN   17.60   17.60   17.60   17.60   17.60
128          1.0   62.58  NaN   62.58   62.58   62.58   62.58   62.58
256          1.0  215.69  NaN  215.69  215.69  215.69  215.69  215.69
512          1.0   30.23  NaN   30.23   30.23   30.23   30.23   30.23


## 2. Network-Level Metrics: `flows_added` and `total_hops`

How do these metrics grow with NPU count and workload parameters?

In [164]:
# --- Plot: flows_added and total_hops vs NPU count (log-log) ---

metrics = ['flows_added', 'total_hops', 'get_next_messages_calls']
metric_labels = ['Flows Added', 'Total Hops', 'get_next_messages calls']

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=metric_labels,
)

for col_idx, (metric, label) in enumerate(zip(metrics, metric_labels), start=1):
    agg = df.groupby('npu_count', as_index=False)[metric].mean()
    agg = agg.sort_values('npu_count')

    # Power-law fit
    x_log = np.log10(agg['npu_count'].values)
    y_log = np.log10(agg[metric].values.clip(min=1e-9))
    if len(x_log) >= 2:
        coeffs = np.polyfit(x_log, y_log, 1)
        exponent = coeffs[0]
        x_fit = np.linspace(x_log.min(), x_log.max(), 50)
        y_fit = np.polyval(coeffs, x_fit)
        fig.add_trace(go.Scatter(
            x=10**x_fit, y=10**y_fit,
            mode='lines', name=f'fit α={exponent:.2f}',
            line=dict(dash='dash', width=2, color='red'),
            showlegend=(col_idx == 1),
        ), row=1, col=col_idx)

    fig.add_trace(go.Scatter(
        x=agg['npu_count'], y=agg[metric],
        mode='markers+lines', name=label,
        marker=dict(size=9, color='skyblue', line=dict(width=1, color='black')),
        showlegend=False,
    ), row=1, col=col_idx)

    fig.update_xaxes(title_text='NPU Count', type='log',
                     tickvals=[2, 4, 8, 16, 32, 64, 128, 256, 512, 1024], row=1, col=col_idx)
    fig.update_yaxes(title_text=label, type='log', row=1, col=col_idx)

fig.update_layout(
    title='G2 Network Metrics vs NPU Count (log-log, mean across workloads)',
    template='plotly_white', font=dict(size=13),
    height=450, width=1200,
)
fig.show()

In [165]:
# --- Plot: flows_added and total_hops vs workload params (2x2 per metric) ---

param_axis = {
    'd_model': 'Model Dimension',
    'num_stacks': 'Num Layers',
    'seq_len': 'Sequence Length',
    'batch': 'Batch Size',
}

for metric, mlabel in [('flows_added', 'Flows Added'), ('total_hops', 'Total Hops')]:
    fig = make_subplots(rows=2, cols=2, subplot_titles=list(param_axis.values()))
    for idx, (param, plabel) in enumerate(param_axis.items()):
        row, col = divmod(idx, 2)
        row += 1; col += 1
        agg = df.groupby(param, as_index=False)[metric].mean().sort_values(param)
        fig.add_trace(go.Scatter(
            x=agg[param], y=agg[metric],
            mode='lines+markers',
            marker=dict(size=7, color='skyblue'),
            line=dict(color='skyblue', width=2),
            showlegend=False,
        ), row=row, col=col)
        fig.update_xaxes(title_text=plabel, row=row, col=col)
        fig.update_yaxes(title_text=mlabel, type='log', row=row, col=col)

    fig.update_layout(
        title=f'G2 {mlabel} vs Workload Parameters',
        template='plotly_white', font=dict(size=12),
        height=650, width=950,
    )
    fig.show()

## 3. Correlation: What Drives G2 Tracked Time?

Spearman correlations between workload/network metrics and G2 **tracked** time (excluding AstraSim overhead).

In [166]:
# --- Spearman correlation: network metrics vs tracked_s (G2-only time) ---

corr_cols = [
    'npu_count', 'flows_added', 'total_hops', 'get_next_messages_calls',
    'remove_messages_calls', 'peak_transmitting_flows', 'peak_active_flow_routes',
    'json_recreations', 'threshold_recreates',
    'd_model', 'num_stacks', 'seq_len', 'batch',
    'tracked_s',
]
corr_labels = [
    'NPU count', 'flows_added', 'total_hops', 'get_next_msgs',
    'remove_msgs', 'peak_tx_flows', 'peak_active_routes',
    'json_recreations', 'threshold_recreates',
    'd_model', 'num_stacks', 'seq_len', 'batch',
    'G2 tracked time',
]

sdf = df.dropna(subset=corr_cols)
n = len(corr_cols)
corr_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        r, _ = spearmanr(sdf[corr_cols[i]], sdf[corr_cols[j]])
        corr_matrix[i, j] = r

fig = go.Figure(go.Heatmap(
    z=corr_matrix, x=corr_labels, y=corr_labels,
    colorscale='RdBu', zmid=0, zmin=-1, zmax=1,
    text=np.round(corr_matrix, 2), texttemplate='%{text}',
    colorbar=dict(title='Spearman ρ'),
))
fig.update_layout(
    title='Spearman Correlation: G2 Network Metrics & Workload Params vs Tracked Time',
    template='plotly_white', font=dict(size=11),
    height=650, width=800,
    xaxis=dict(tickangle=-45),
)
fig.show()

In [167]:
# --- Per-call avg_ms for top functions vs workload params ---
# More useful than raw flows vs wall-time: shows which params make individual calls expensive

et_noinit = et_df[et_df['section'] != 'Init function time'].copy()

param_axis = {
    'npu_count': 'NPU Count',
    'peak_active_flow_routes': 'Peak Active Flow Routes',
}

for metric_col, metric_label in [('avg_ms', 'Avg ms per call')]:
    fig = make_subplots(rows=1, cols=2, subplot_titles=list(param_axis.values()))
    # Pick top 5 costliest functions by mean avg_ms
    top5_expensive = (
        et_noinit.groupby('section', as_index=False)['avg_ms'].mean()
        .sort_values('avg_ms', ascending=False).head(5)['section'].tolist()
    )
    colors = px.colors.qualitative.Plotly[:5]
    for col_idx, (param, plabel) in enumerate(param_axis.items(), start=1):
        for i, sec in enumerate(top5_expensive):
            if param in et_noinit.columns:
                sdf_sec = et_noinit[et_noinit['section'] == sec]
            else:
                sdf_sec = et_noinit[et_noinit['section'] == sec].merge(
                    df[['workload', 'npu_count', 'run', param]], on=['workload', 'npu_count', 'run'])
            agg = sdf_sec.groupby(param, as_index=False)[metric_col].mean().sort_values(param)
            fig.add_trace(go.Scatter(
                x=agg[param], y=agg[metric_col],
                mode='lines+markers', name=sec,
                marker=dict(size=6, color=colors[i]),
                line=dict(color=colors[i], width=2),
                showlegend=(col_idx == 1),
            ), row=1, col=col_idx)
        fig.update_xaxes(title_text=plabel, type='log', row=1, col=col_idx)
        fig.update_yaxes(title_text=metric_label, type='log', row=1, col=col_idx)

    fig.update_layout(
        title='Per-Call Cost of Top-5 Expensive Functions vs NPU Count & Peak Concurrency',
        template='plotly_white', font=dict(size=12),
        height=450, width=1100,
        legend=dict(font=dict(size=9)),
    )
    fig.show()

## 4. Function-Level Time Breakdown

Which internal functions dominate G2 wall-clock time, and how does their share change with NPU count?

In [168]:
# --- Top functions by total time (aggregated across all runs) ---

# Exclude 'Init function time' as it's a one-time startup cost
et_noinit = et_df[et_df['section'] != 'Init function time'].copy()

top_sections = (
    et_noinit
    .groupby('section', as_index=False)['total_s']
    .sum()
    .sort_values('total_s', ascending=False)
    .head(15)
)

fig = go.Figure(go.Bar(
    y=top_sections['section'],
    x=top_sections['total_s'],
    orientation='h',
    marker_color='skyblue',
))
fig.update_layout(
    title='Top 15 G2 Functions by Total Time (all runs combined, excl. Init)',
    xaxis_title='Total Time (s)',
    yaxis=dict(autorange='reversed'),
    template='plotly_white', font=dict(size=12),
    height=550, width=950,
    margin=dict(l=350),
)
fig.show()

In [169]:
# --- Stacked bar: % of tracked time per function, grouped by NPU count ---

# Pick top 15 sections by total time
top15_names = top_sections['section'].head(15).tolist()

# Compute mean pct per section per npu_count
pct_by_npu = (
    et_noinit[et_noinit['section'].isin(top15_names)]
    .groupby(['npu_count', 'section'], as_index=False)['pct']
    .mean()
)

fig = go.Figure()
colors = px.colors.qualitative.Set3[:len(top15_names)]
for i, sec in enumerate(top15_names):
    sdf = pct_by_npu[pct_by_npu['section'] == sec].sort_values('npu_count')
    fig.add_trace(go.Bar(
        x=sdf['npu_count'].astype(str), y=sdf['pct'],
        name=sec, marker_color=colors[i % len(colors)],
    ))

fig.update_layout(
    barmode='stack',
    title='Share of Tracked Time per Top-15 Functions by NPU Count',
    xaxis_title='NPU Count', yaxis_title='Mean % of Tracked Time',
    template='plotly_white', font=dict(size=12),
    height=550, width=1050,
    legend=dict(font=dict(size=9)),
)
fig.show()

In [170]:
# --- Line plots: absolute time per function vs NPU count (log-log) ---
# Shows how each function's total_s scales with NPU count

fig = go.Figure()
colors = px.colors.qualitative.Plotly + px.colors.qualitative.Set2

for i, sec in enumerate(top10_names):
    agg = (
        et_noinit[et_noinit['section'] == sec]
        .groupby('npu_count', as_index=False)['total_s']
        .mean()
        .sort_values('npu_count')
    )
    if len(agg) < 2:
        continue

    # Power-law fit
    xl = np.log10(agg['npu_count'].values)
    yl = np.log10(agg['total_s'].values.clip(min=1e-9))
    alpha = np.polyfit(xl, yl, 1)[0]

    fig.add_trace(go.Scatter(
        x=agg['npu_count'], y=agg['total_s'],
        mode='lines+markers',
        name=f'{sec} (α={alpha:.2f})',
        marker=dict(size=7),
        line=dict(color=colors[i % len(colors)], width=2),
    ))

fig.update_layout(
    title='G2 Function Mean Time vs NPU Count (log-log) — Power-Law Exponents',
    xaxis_title='NPU Count', yaxis_title='Mean Total Time (s)',
    xaxis_type='log', yaxis_type='log',
    xaxis=dict(tickvals=[2, 4, 8, 16, 32, 64]),
    template='plotly_white', font=dict(size=12),
    height=550, width=1100,
    legend=dict(font=dict(size=9)),
)
fig.show()

In [171]:
# --- Table: scaling exponents per function ---

scaling_rows = []
for sec in et_noinit['section'].unique():
    agg = (
        et_noinit[et_noinit['section'] == sec]
        .groupby('npu_count', as_index=False)
        .agg(mean_total_s=('total_s', 'mean'), mean_calls=('calls', 'mean'), mean_pct=('pct', 'mean'))
        .sort_values('npu_count')
    )
    if len(agg) < 2 or agg['mean_total_s'].max() < 1e-6:
        continue
    xl = np.log10(agg['npu_count'].values)
    yl_time = np.log10(agg['mean_total_s'].values.clip(min=1e-12))
    yl_calls = np.log10(agg['mean_calls'].values.clip(min=1e-12))
    alpha_time = np.polyfit(xl, yl_time, 1)[0]
    alpha_calls = np.polyfit(xl, yl_calls, 1)[0]
    scaling_rows.append({
        'section': sec,
        'α_time (NPU exponent)': round(alpha_time, 3),
        'α_calls (NPU exponent)': round(alpha_calls, 3),
        'mean_pct (all runs)': round(agg['mean_pct'].mean(), 2),
        'total_s (64 NPUs)': round(agg[agg['npu_count'] == 64]['mean_total_s'].values[0], 4) if 64 in agg['npu_count'].values else None,
    })

scaling_df = pd.DataFrame(scaling_rows).sort_values('α_time (NPU exponent)', ascending=False)
print('Function scaling exponents (power-law fit log10(time) ~ α * log10(npu_count)):')
display(scaling_df.head(20).style.format(precision=3).background_gradient(
    subset=['α_time (NPU exponent)'], cmap='Reds'
).background_gradient(
    subset=['mean_pct (all runs)'], cmap='Blues'
))

Function scaling exponents (power-law fit log10(time) ~ α * log10(npu_count)):


,section,α_time (NPU exponent),α_calls (NPU exponent),mean_pct (all runs),total_s (64 NPUs)
2,[remove_messages] _bulk_update_flowgroups Superset transform,3.151,1.775,10.950,3.562
19,[remove_messages] _bulk_update_flowgroups Superset lookup,2.761,1.761,0.500,0.285
31,[remove_messages] _bulk_update_flowgroups Sequential updates,2.623,1.292,4.090,1.204
28,[get_next_messages] _bulk_update_flowgroups Superset lookup,2.601,1.686,0.320,0.208
20,[get_next_messages] _bulk_update_flowgroups Sequential updates,2.507,1.595,14.020,4.522
8,[get_next_messages] _bulk_update_flowgroups Superset transform,1.930,0.681,3.120,0.351
16,Remove messages time update transmitting flows,1.830,1.024,0.690,0.153
0,Get next messages time Rate Loop,1.812,1.151,10.160,1.874
17,Get next messages time Record current state loop,1.796,0.900,0.650,0.138
10,Get next messages flows left Loop,1.779,1.151,1.660,0.300


## 5. Per-Call Latency: Are Individual Calls Getting Slower?

**Key question:** Is G2 slow because there are more calls, or because each call takes longer at higher NPU counts?  
This is the most actionable section — functions where `avg_ms` grows with NPU count are optimization targets.

In [172]:
# --- avg_ms per call vs NPU count for top functions ---

fig = go.Figure()
colors = px.colors.qualitative.Plotly + px.colors.qualitative.Set2

for i, sec in enumerate(top10_names):
    agg = (
        et_noinit[et_noinit['section'] == sec]
        .groupby('npu_count', as_index=False)['avg_ms']
        .mean()
        .sort_values('npu_count')
    )
    if len(agg) < 2:
        continue

    xl = np.log10(agg['npu_count'].values)
    yl = np.log10(agg['avg_ms'].values.clip(min=1e-12))
    alpha = np.polyfit(xl, yl, 1)[0]

    fig.add_trace(go.Scatter(
        x=agg['npu_count'], y=agg['avg_ms'],
        mode='lines+markers',
        name=f'{sec} (α={alpha:.2f})',
        marker=dict(size=7),
        line=dict(color=colors[i % len(colors)], width=2),
    ))

fig.update_layout(
    title='G2 Per-Call Latency (avg_ms) vs NPU Count — Are Calls Getting Slower?',
    xaxis_title='NPU Count', yaxis_title='Mean avg_ms per call',
    xaxis_type='log', yaxis_type='log',
    xaxis=dict(tickvals=[2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]),
    template='plotly_white', font=dict(size=12),
    height=500, width=1100,
    legend=dict(font=dict(size=9)),
)
fig.show()

# --- Heatmap: avg_ms per function × NPU count ---
# Visual matrix showing which functions become expensive at scale

heat_data = (
    et_noinit[et_noinit['section'].isin(top10_names)]
    .groupby(['section', 'npu_count'], as_index=False)['avg_ms']
    .mean()
    .pivot(index='section', columns='npu_count', values='avg_ms')
    .reindex(top10_names)  # keep same order as top10
)

fig2 = go.Figure(go.Heatmap(
    z=np.log10(heat_data.values.clip(min=1e-12)),
    x=[str(c) for c in heat_data.columns],
    y=heat_data.index,
    colorscale='YlOrRd',
    text=heat_data.values.round(4),
    texttemplate='%{text:.4f}',
    colorbar=dict(title='log₁₀(avg_ms)'),
))
fig2.update_layout(
    title='Per-Call Cost Heatmap: avg_ms by Function × NPU Count',
    xaxis_title='NPU Count', yaxis_title='Function',
    template='plotly_white', font=dict(size=11),
    height=450, width=900,
    margin=dict(l=350),
)
fig2.show()

In [173]:
# --- Decomposition: total_time = calls × avg_time_per_call ---
# Show which factor dominates growth for each function (not just top10)

all_sections = et_noinit['section'].unique()
decomp_rows = []
for sec in all_sections:
    agg = (
        et_noinit[et_noinit['section'] == sec]
        .groupby('npu_count', as_index=False)
        .agg(mean_total_s=('total_s', 'mean'), mean_calls=('calls', 'mean'), mean_avg_ms=('avg_ms', 'mean'))
        .sort_values('npu_count')
    )
    if len(agg) < 2 or agg['mean_total_s'].max() < 1e-6:
        continue
    xl = np.log10(agg['npu_count'].values)
    alpha_time = np.polyfit(xl, np.log10(agg['mean_total_s'].values.clip(min=1e-12)), 1)[0]
    alpha_calls = np.polyfit(xl, np.log10(agg['mean_calls'].values.clip(min=1e-12)), 1)[0]
    alpha_per_call = np.polyfit(xl, np.log10(agg['mean_avg_ms'].values.clip(min=1e-12)), 1)[0]

    # Get avg_ms at smallest and largest NPU count
    avg_ms_min_npu = agg.iloc[0]['mean_avg_ms']
    avg_ms_max_npu = agg.iloc[-1]['mean_avg_ms']

    decomp_rows.append({
        'Function': sec,
        'α(total_time)': round(alpha_time, 3),
        'α(num_calls)': round(alpha_calls, 3),
        'α(per_call_ms)': round(alpha_per_call, 3),
        f'avg_ms @{int(agg.iloc[0]["npu_count"])}NPU': round(avg_ms_min_npu, 5),
        f'avg_ms @{int(agg.iloc[-1]["npu_count"])}NPU': round(avg_ms_max_npu, 5),
        'slowdown_factor': round(avg_ms_max_npu / max(avg_ms_min_npu, 1e-12), 1),
        'Dominant growth': 'more calls' if alpha_calls > alpha_per_call else 'slower calls',
    })

decomp_df = pd.DataFrame(decomp_rows).sort_values('α(total_time)', ascending=False)
print('Time growth decomposition: α(total) ≈ α(calls) + α(per_call)')
print('  - α(calls) >> α(per_call) → slowness from more calls (workload-driven)')
print('  - α(per_call) >> α(calls) → each call gets slower (algorithmic issue)')
print(f'  - slowdown_factor = avg_ms at max NPU / avg_ms at min NPU\n')
display(decomp_df.head(20).style.format(precision=3).background_gradient(
    subset=['α(total_time)', 'α(per_call_ms)'], cmap='Reds'
).background_gradient(
    subset=['slowdown_factor'], cmap='YlOrRd'
))

Time growth decomposition: α(total) ≈ α(calls) + α(per_call)
  - α(calls) >> α(per_call) → slowness from more calls (workload-driven)
  - α(per_call) >> α(calls) → each call gets slower (algorithmic issue)
  - slowdown_factor = avg_ms at max NPU / avg_ms at min NPU



,Function,α(total_time),α(num_calls),α(per_call_ms),avg_ms @2NPU,avg_ms @512NPU,slowdown_factor,Dominant growth,avg_ms @32NPU,avg_ms @4NPU
2,[remove_messages] _bulk_update_flowgroups Superset transform,3.151,1.775,1.375,0.008,4.070,538.700,more calls,nan,nan
19,[remove_messages] _bulk_update_flowgroups Superset lookup,2.761,1.761,1.000,0.003,0.089,34.200,more calls,nan,nan
31,[remove_messages] _bulk_update_flowgroups Sequential updates,2.623,1.292,1.331,0.017,7.137,423.800,slower calls,nan,nan
28,[get_next_messages] _bulk_update_flowgroups Superset lookup,2.601,1.686,0.915,0.003,0.071,26.600,more calls,nan,nan
20,[get_next_messages] _bulk_update_flowgroups Sequential updates,2.507,1.595,0.912,0.099,5.997,60.300,more calls,nan,nan
8,[get_next_messages] _bulk_update_flowgroups Superset transform,1.930,0.681,1.249,nan,4.916,18.100,slower calls,0.272,nan
16,Remove messages time update transmitting flows,1.830,1.024,0.806,0.001,0.018,14.500,more calls,nan,nan
0,Get next messages time Rate Loop,1.812,1.151,0.661,0.008,0.130,15.700,more calls,nan,nan
17,Get next messages time Record current state loop,1.796,0.900,0.897,0.001,0.029,21.800,more calls,nan,nan
10,Get next messages flows left Loop,1.779,1.151,0.628,0.002,0.020,12.900,more calls,nan,nan


## 6. Cache Efficiency

How do cache hit rates and JSON recreations change with scale?

In [174]:
# --- Cache metrics vs NPU count ---

df['cache_hit_rate'] = df['cache_hits'] / (df['cache_hits'] + df['json_recreations']).clip(lower=1)
df['superset_hit_rate'] = df['superset_hits'] / (df['superset_hits'] + df['superset_misses']).clip(lower=1)

fig = make_subplots(rows=1, cols=3,
    subplot_titles=['Cache Hit Rate', 'Superset Hit Rate', 'JSON Recreations'])

agg1 = df.groupby('npu_count', as_index=False)['cache_hit_rate'].mean().sort_values('npu_count')
fig.add_trace(go.Scatter(x=agg1['npu_count'], y=agg1['cache_hit_rate'] * 100,
    mode='lines+markers', marker=dict(size=8, color='green'), showlegend=False), row=1, col=1)
fig.update_xaxes(title_text='NPU Count', type='log', row=1, col=1)
fig.update_yaxes(title_text='Hit Rate (%)', row=1, col=1)

agg2 = df.groupby('npu_count', as_index=False)['superset_hit_rate'].mean().sort_values('npu_count')
fig.add_trace(go.Scatter(x=agg2['npu_count'], y=agg2['superset_hit_rate'] * 100,
    mode='lines+markers', marker=dict(size=8, color='blue'), showlegend=False), row=1, col=2)
fig.update_xaxes(title_text='NPU Count', type='log', row=1, col=2)
fig.update_yaxes(title_text='Hit Rate (%)', row=1, col=2)

agg3 = df.groupby('npu_count', as_index=False)['json_recreations'].mean().sort_values('npu_count')
fig.add_trace(go.Scatter(x=agg3['npu_count'], y=agg3['json_recreations'],
    mode='lines+markers', marker=dict(size=8, color='orange'), showlegend=False), row=1, col=3)
fig.update_xaxes(title_text='NPU Count', type='log', row=1, col=3)
fig.update_yaxes(title_text='JSON Recreations', type='log', row=1, col=3)

fig.update_layout(
    title='G2 Cache Efficiency vs NPU Count',
    template='plotly_white', font=dict(size=12),
    height=400, width=1100,
)
fig.show()

## 7. Multi-Variable Regression: What Drives G2 Tracked Time?

Fit `log10(tracked_s) ~ a*log10(flows_added) + b*log10(total_hops) + c*log10(npu_count) + ...`  
Uses only G2 tracked time (excludes AstraSim overhead).

In [175]:
# --- Log-linear regression with network metrics as features (tracked_s only) ---

# Add log_tracked_s
df['log_tracked_s'] = np.log10(df['tracked_s'].clip(lower=eps))

reg_features_raw = ['npu_count', 'flows_added', 'total_hops', 'get_next_messages_calls',
                    'json_recreations', 'd_model', 'seq_len', 'batch']
reg_features = [f'log_{c}' for c in reg_features_raw]
target = 'log_tracked_s'

# Build log features that might not exist yet
for c in reg_features_raw:
    lc = f'log_{c}'
    if lc not in df.columns:
        df[lc] = np.log10(df[c].clip(lower=eps))

sdf = df.dropna(subset=reg_features + [target])
mask = np.isfinite(sdf[reg_features + [target]]).all(axis=1)
sdf = sdf[mask]

X = sdf[reg_features].values
y = sdf[target].values

model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)
r2 = model.score(X, y)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=y, y=y_pred, mode='markers',
    marker=dict(color='skyblue', size=6, opacity=0.7),
))
mn, mx = min(y.min(), y_pred.min()), max(y.max(), y_pred.max())
fig.add_trace(go.Scatter(x=[mn, mx], y=[mn, mx], mode='lines', line=dict(color='black', dash='dot'), name='perfect'))
fig.update_layout(
    title=f'G2: Predicted vs Actual log₁₀(tracked_time) — R²={r2:.4f}',
    xaxis_title='Actual log₁₀(tracked_s)', yaxis_title='Predicted log₁₀(tracked_s)',
    template='plotly_white', font=dict(size=13), height=480, width=600,
)
fig.show()

print(f'\nR² = {r2:.4f}')
print(f'Intercept: {model.intercept_:.4f}')
for feat, coef in zip(reg_features_raw, model.coef_):
    print(f'  log({feat:>28s})  coef = {coef:+.4f}')


R² = 1.0000
Intercept: 3.9552
  log(                   npu_count)  coef = +1.0785
  log(                 flows_added)  coef = -4.0879
  log(                  total_hops)  coef = +4.9240
  log(     get_next_messages_calls)  coef = -0.2060
  log(            json_recreations)  coef = -0.1483
  log(                     d_model)  coef = -1.9883
  log(                     seq_len)  coef = -1.0557
  log(                       batch)  coef = -0.2452


## 8. Hops per Flow & Peak Concurrency

Derived metrics that highlight structural complexity.

In [176]:
# --- Derived metrics ---

df['hops_per_flow'] = df['total_hops'] / df['flows_added'].clip(lower=1)
df['calls_per_flow'] = df['get_next_messages_calls'] / df['flows_added'].clip(lower=1)

fig = make_subplots(rows=1, cols=3,
    subplot_titles=['Hops per Flow', 'Calls per Flow', 'Peak Active Flow Routes'])

for col_idx, (metric, label) in enumerate([
    ('hops_per_flow', 'Hops / Flow'),
    ('calls_per_flow', 'get_next_msgs / Flow'),
    ('peak_active_flow_routes', 'Peak Active Routes'),
], start=1):
    agg = df.groupby('npu_count', as_index=False)[metric].mean().sort_values('npu_count')
    fig.add_trace(go.Scatter(
        x=agg['npu_count'], y=agg[metric],
        mode='lines+markers', marker=dict(size=8, color='purple'),
        showlegend=False,
    ), row=1, col=col_idx)
    fig.update_xaxes(title_text='NPU Count', type='log',
                     tickvals=[2, 4, 8, 16, 32, 64], row=1, col=col_idx)
    fig.update_yaxes(title_text=label, row=1, col=col_idx)

fig.update_layout(
    title='G2 Derived Metrics vs NPU Count',
    template='plotly_white', font=dict(size=12),
    height=400, width=1100,
)
fig.show()

## 9. Marginal Cost per Flow / Hop

Normalize G2 tracked time by flows and hops to see if the per-unit cost grows with scale.

In [177]:
# --- Tracked time normalized by flows / hops (excludes AstraSim overhead) ---

df['tracked_ms_per_flow'] = 1000 * df['tracked_s'] / df['flows_added'].clip(lower=1)
df['tracked_ms_per_hop'] = 1000 * df['tracked_s'] / df['total_hops'].clip(lower=1)
df['tracked_ms_per_gnm_call'] = 1000 * df['tracked_s'] / df['get_next_messages_calls'].clip(lower=1)

fig = make_subplots(rows=1, cols=3,
    subplot_titles=['Tracked ms / Flow', 'Tracked ms / Hop', 'Tracked ms / get_next_msg call'])

for col_idx, (metric, label) in enumerate([
    ('tracked_ms_per_flow', 'ms / flow'),
    ('tracked_ms_per_hop', 'ms / hop'),
    ('tracked_ms_per_gnm_call', 'ms / call'),
], start=1):
    for npu in sorted(df.npu_count.unique()):
        sdf = df[df.npu_count == npu]
        fig.add_trace(go.Box(
            y=sdf[metric], name=str(npu),
            marker_color='skyblue',
            showlegend=False,
        ), row=1, col=col_idx)
    fig.update_yaxes(title_text=label, type='log', row=1, col=col_idx)
    fig.update_xaxes(title_text='NPU Count', row=1, col=col_idx)

fig.update_layout(
    title='G2 Marginal Cost: Tracked Time Normalized by Flows / Hops / Calls',
    template='plotly_white', font=dict(size=12),
    height=450, width=1200,
)
fig.show()

## 10. Summary Table

Compact overview of key metrics per NPU tier (G2 tracked time only).

In [178]:
# --- Summary table (tracked time only, no AstraSim overhead) ---

summary = df.groupby('npu_count').agg(
    n_runs=('workload', 'count'),
    mean_tracked_s=('tracked_s', 'mean'),
    median_tracked_s=('tracked_s', 'median'),
    max_tracked_s=('tracked_s', 'max'),
    mean_flows=('flows_added', 'mean'),
    mean_hops=('total_hops', 'mean'),
    mean_gnm_calls=('get_next_messages_calls', 'mean'),
    mean_peak_routes=('peak_active_flow_routes', 'mean'),
    mean_json_recreations=('json_recreations', 'mean'),
    mean_hops_per_flow=('hops_per_flow', 'mean'),
    mean_tracked_ms_per_flow=('tracked_ms_per_flow', 'mean'),
).round(2)

print('G2 Performance Summary by NPU Count (tracked time only):')
display(summary.style.format(precision=2).background_gradient(cmap='YlOrRd'))

G2 Performance Summary by NPU Count (tracked time only):


,n_runs,mean_tracked_s,median_tracked_s,max_tracked_s,mean_flows,mean_hops,mean_gnm_calls,mean_peak_routes,mean_json_recreations,mean_hops_per_flow,mean_tracked_ms_per_flow
npu_count,,,,,,,,,,,
2,1,0.01,0.01,0.01,8.00,16.00,245.00,2.00,0.00,2.00,1.60
4,1,1.28,1.28,1.28,36912.00,73824.00,46084.00,4.00,1.00,2.00,0.03
8,1,0.05,0.05,0.05,768.00,1536.00,317.00,8.00,1.00,2.00,0.07
16,1,0.18,0.18,0.18,2704.00,5440.00,1083.00,16.00,6.00,2.01,0.07
32,1,8.26,8.26,8.26,132352.00,412928.00,97700.00,32.00,3.00,3.12,0.06
64,1,14.96,14.96,14.96,28288.00,69472.00,33037.00,64.00,6.00,2.46,0.53
128,1,25.67,25.67,25.67,513536.00,1779712.00,49526.00,128.00,4.00,3.47,0.05
256,1,193.33,193.33,193.33,28672.00,81600.00,52016.00,256.00,11.00,2.85,6.74
512,1,26.78,26.78,26.78,76928.00,202624.00,25567.00,128.00,151.00,2.63,0.35
